In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from telethon.sync import TelegramClient
import csv
import asyncio
from telethon import TelegramClient, events
from telethon.tl.functions.channels import GetFullChannelRequest
from datetime import datetime, timedelta
import csv
from telethon import TelegramClient, events, errors


In [ ]:
api_id = 27006778
api_hash = 'f41204fef3102a1ca48d248f3c287425'
phone = '+251924246518'
channels = [
    'ZemenExpress',
    'Leyueqa',
    'MerttEka',
    'classybrands',
    'belaclassic',
    'AwasMart'
]

client = TelegramClient('my_session', api_id, api_hash)

# Define the date range for messages to be collected
start_date = datetime.now() - timedelta(days=2)
end_date = datetime.now()

await client.start()
with open('telegram_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Channel', 'Timestamp', 'Message', 'Views'])

    @client.on(events.NewMessage(channels))
    async def handler(event):
         if start_date <= event.message.date <= end_date:
            channel = event.chat.title or event.chat.username
            timestamp = event.message.date
            message = event.message.message or ''
            views = event.message.views or 0

            writer.writerow([channel, timestamp, message, views])
            print(f'New message from {channel}: {message}')

try:
     await client.run_until_disconnected()
except errors.SessionRevokedError:
    print("Session has been revoked. Please reauthorize your session.")

In [ ]:
from telethon import TelegramClient
import pandas as pd

# Initialize the Telegram client
api_id = 27006778
api_hash = 'f41204fef3102a1ca48d248f3c287425'
client = TelegramClient('session_name', api_id, api_hash)

async def fetch_messages(channel):
    await client.start()
    messages = await client.get_messages(channel, limit=100)
    return [message.text for message in messages if message.text]

async def main():
    channels = ['ZemenExpress','Leyueqa','MerttEka','classybrands','belaclassic','AwasMart'
]
    all_messages = []
    for channel in channels:
        messages = await fetch_messages(channel)
        all_messages.extend(messages)
    return all_messages

messages = client.loop.run_until_complete(main())

RuntimeError: 'run_cell_async' needs a real async loop

In [ ]:
import nltk
import re

nltk.download('punkt')

def preprocess_text(text):
    # Remove unwanted characters
    text = re.sub(r'[^ሀ-ፈ 0-9a-zA-Z]+', ' ', text)
    # Tokenize the text
    tokens = nltk.word_tokenize(text, language='amharic')
    # Normalize (e.g., convert to lowercase)
    normalized_tokens = [token.lower() for token in tokens]
    return ' '.join(normalized_tokens)

cleaned_messages = [preprocess_text(msg) for msg in messages]

In [ ]:
data = {
    'Channel': ['channel1'] * len(cleaned_messages),  # Adjust accordingly
    'Cleaned_Message': cleaned_messages,
}
df = pd.DataFrame(data)
df.to_csv('telegram_data.csv', index=False)

In [ ]:
def label_entities(message):
    tokens = message.split()
    labels = []
    
    for token in tokens:
        if 'ዋጋ' in token or 'ብር' in token:
            labels.append('B-PRICE' if 'ዋጋ' in token else 'I-PRICE')
        elif token in ["addis", "abeba", "bole"]:
            labels.append('B-LOC' if token == "addis" else 'I-LOC')
        elif "Baby" in token:
            labels.append('B-Product')
        else:
            labels.append('O')
    
    return list(zip(tokens, labels))

In [ ]:
def prepare_conll_format(df):
    conll_lines = []
    for _, row in df.iterrows():
        message = row['Cleaned_Message']
        labeled_entities = label_entities(message)
        
        for token, label in labeled_entities:
            conll_lines.append(f"{token} {label}")
        
        conll_lines.append("")  # Blank line to separate messages

    return "\n".join(conll_lines)

conll_output = prepare_conll_format(df)
with open('labeled_data.conll', 'w', encoding='utf-8') as f:
    f.write(conll_output)